# Récupération du dataset USDA — Détection de défauts sur pommes de terre

Ce notebook télécharge le jeu de données de Feldman et al. (2024) hébergé sur l'**USDA Ag
Data Commons** : images RGB de pommes de terre coupées pour la détection du "cœur creux"
(hollow heart), plus les images de tubercules entiers (taille, forme, couleur).

DOI : `10.15482/USDA.ADC/1529451`

Conçu pour tourner sur Google Colab ou en local.


In [1]:
!pip install requests tqdm pillow matplotlib -q

In [2]:
import os
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm

# Dossier racine où seront stockées toutes les images téléchargées
BASE_DIR = Path("dataset")
BASE_DIR.mkdir(exist_ok=True)
print(f"Répertoire de travail : {BASE_DIR.resolve()}")

Répertoire de travail : C:\Users\LBS\STAGE HANOI - HUST\dataset


## USDA Ag Data Commons — Feldman et al. (2024)

L'Ag Data Commons de l'USDA (NAL) repose sur la plateforme **CKAN**. On interroge son API
pour retrouver le jeu de données à partir de son DOI, puis on télécharge chaque ressource
associée (images, archives zip, etc.).

⚠️ Cette recherche automatique dépend de la façon dont l'USDA a indexé le DOI dans CKAN.
Si elle échoue, un lien de repli vers la page DOI est affiché pour un téléchargement manuel.

In [7]:
def find_usda_adc_dataset_by_doi(doi):
    """
    Interroge l'API CKAN de l'Ag Data Commons pour retrouver le jeu de données
    correspondant à un DOI donné.
    """
    api_search = "https://data.nal.usda.gov/api/3/action/package_search"
    params = {"q": f'doi:"{doi}"'}
    resp = requests.get(api_search, params=params, timeout=30)
    resp.raise_for_status()
    results = resp.json()["result"]["results"]
    if not results:
        raise ValueError("Aucun jeu de données trouvé automatiquement pour ce DOI.")
    return results[0]

doi = "10.15482/USDA.ADC/1529451"
dataset = None
resources = []

try:
    dataset = find_usda_adc_dataset_by_doi(doi)
    resources = dataset["resources"]
    print("Jeu de données trouvé :", dataset["title"])
    for r in resources:
        print(f"  - {r['name']} ({r.get('format', '?')}) -> {r['url']}")
except Exception as e:
    print("La recherche automatique a échoué :", e)
    print("Solution de repli : ouvre ce lien dans un navigateur et télécharge les fichiers à la main :")
    print(f"https://doi.org/{doi}")

La recherche automatique a échoué : Expecting value: line 1 column 1 (char 0)
Solution de repli : ouvre ce lien dans un navigateur et télécharge les fichiers à la main :
https://doi.org/10.15482/USDA.ADC/1529451


In [4]:
usda_dest = BASE_DIR / "usda_potato_defects"
usda_dest.mkdir(parents=True, exist_ok=True)

for r in resources:
    url = r["url"]
    fname = url.split("/")[-1] or r["name"]
    out_path = usda_dest / fname
    print(f"Téléchargement de {fname} ...")
    try:
        resp = requests.get(url, stream=True, timeout=60)
        resp.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        # Dézippe automatiquement si c'est une archive
        if fname.lower().endswith(".zip"):
            with zipfile.ZipFile(out_path, "r") as z:
                z.extractall(usda_dest)
            print(f"  -> Archive décompressée dans {usda_dest}")
    except Exception as e:
        print(f"  Échec du téléchargement de {fname} : {e}")

if not resources:
    print("Aucune ressource à télécharger (voir la cellule précédente pour le lien manuel).")

Aucune ressource à télécharger (voir la cellule précédente pour le lien manuel).


## Bilan et vérification rapide

Compte le nombre d'images récupérées et affiche un échantillon aléatoire pour vérifier
visuellement que le téléchargement s'est bien passé.

In [5]:
import random
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")

def show_sample(folder, n=6):
    folder = Path(folder)
    if not folder.exists():
        print(f"{folder} n'existe pas encore.")
        return
    files = [f for f in folder.glob("*") if f.suffix.lower() in IMAGE_EXT]
    if not files:
        print(f"Aucune image trouvée dans {folder}")
        return
    sample = random.sample(files, min(n, len(files)))
    fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
    if len(sample) == 1:
        axes = [axes]
    for ax, f in zip(axes, sample):
        img = Image.open(f)
        ax.imshow(img)
        ax.set_title(f.name, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Résumé des fichiers récupérés :")
for folder in sorted(BASE_DIR.iterdir()):
    if folder.is_dir():
        n_files = sum(1 for _ in folder.rglob("*") if _.is_file())
        print(f"  {folder.name}: {n_files} fichier(s)")

# Exemple d'utilisation :
# show_sample(BASE_DIR / "tubar_samples")
# show_sample(BASE_DIR / "usda_potato_defects")

Résumé des fichiers récupérés :
  usda_potato_defects: 0 fichier(s)


## Prochaine étape

Une fois les images vérifiées visuellement, l'étape suivante est l'augmentation de données
avec `Albumentations` (comme dans Feldman et al.) pour compenser le faible nombre d'images
disponibles avant d'entraîner ton CNN — je peux te préparer ce pipeline si tu veux.